<a href="https://colab.research.google.com/github/emzu/futureIDF/blob/main/01)Process_Timeseries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Initialization

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git config --global user.email "ezuetell@andrew.cmu.edu"
!git config --global user.name "emzu"

try:
    !git clone "https://github.com/emzu/futureIDF"
except:
    print("Already cloned")

%cd /content/futureIDF
!git pull

# Load Packages
!pip install -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path 'futureIDF' already exists and is not an empty directory.
/content/futureIDF
Already up to date.


In [ ]:
!git pull

# Import modules
import sys
import importlib
# Ensure workspace root is on sys.path so the local `modules` package can be imported
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import glob
import os
import warnings
warnings.filterwarnings('ignore')

from modules import config, data_io, timeseries, plotting, process_rp, geospatial
importlib.reload(timeseries)
importlib.reload(data_io)
importlib.reload(process_rp)
importlib.reload(geospatial)

Already up to date.


<module 'modules.geospatial' from '/content/futureIDF/modules/geospatial.py'>

# Download and Process Raw Data (If needed)

##LOCA2
Documentation: https://loca.ucsd.edu/loca-version-2-for-north-america-ca-jan-2023/

Reference: https://doi.org/10.1175/JHM-D-22-0194.1


In [ ]:
from pathlib import Path
import wget


loca2_directory = "https://cirrus.ucsd.edu/~pierce/LOCA2/CONUS_regions_split"

for model in config.MODELS_LOCA2:
  for scen in config.SCENARIOS['LOCA2']:
    for ensemble in range(1,10):
      for time_period in ['1950-2014']:
        ########## DOWNLOAD DATA ##########
        region = "n_east"
        directory = f"{loca2_directory}/{model}/{region}/0p0625deg/r{ensemble}i1p1f1/{scen}/pr"
        file = f"pr.{model}.{scen}.r{ensemble}i1p1f1.{time_period}.LOCA_16thdeg_v20240915.{region}.nc"
        directory_local = Path(f'/content/{model}/{region}/0p0625deg/r{ensemble}i1p1f1/{scen}/pr')
        os.makedirs(directory_local, exist_ok=True)

        file_path_ne= Path(f"{directory_local}/{file}")
        try:
          wget.download(f"{directory}/{file}", f"{directory_local}/{file}")
        except:
          print(f"Skipping {directory_local}/{file}")
          continue



        print(f"Processing {file}...")
        # Open xarray
        ds_ne = xr.open_dataset(f"{directory_local}/{file}")
        # Clip to MARISA domain
        result_ne = geospatial.clip_domain(ds_ne, ver = 'CMIP6')

        region = "s_east"
        directory = f"{loca2_directory}/{model}/{region}/0p0625deg/r{ensemble}i1p1f1/{scen}/pr"
        file = f"pr.{model}.{scen}.r{ensemble}i1p1f1.{time_period}.LOCA_16thdeg_v20240915.{region}.nc"
        directory_local = Path(f'/content/{model}/{region}/0p0625deg/r{ensemble}i1p1f1/{scen}/pr')
        os.makedirs(directory_local, exist_ok=True)

        wget.download(f"{directory}/{file}", f"{directory_local}/{file}")
        file_path_se= Path(f"{directory_local}/{file}")


        print(f"Processing {file}...")
        # Open xarray
        ds_se = xr.open_dataset(f"{directory_local}/{file}")
        # Clip to MARISA domain
        result_se = clip_domain(ds_se)

        #crs = ds_ne.spatial_ref   # save CRS info
        merged = xr.merge([result_ne, result_se])
        result = merged

        ds_se.close()
        ds_ne.close()
        result.to_netcdf(f'/content/{file_path_ne.stem}.nc')
        os.remove(file_path_ne)
        os.remove(file_path_se)

    ########## PROCESS RAW DATA ##########
    DIRECTORY = "/content/drive/MyDrive/Research/MARISA_IDF/data/LOCA2/MARISA"
    scenario = 'historical'
    subset_time = [1950, 1999]
    variable = 'pr'
    try:
      # Load Data, Unit Conversion
      ds = data_io.load_model(
              directory=f"/content",
              model=model,
              scenario = scenario,
              subset_time = subset_time,
              variable = variable,
              pattern = None)

      # Extract Maximums
      result = timeseries.process_precipitation_timeseries(
            ds[variable],
            min_separation_days=7,
            compute_pds=True
        ).compute()

      # Save processed maximums data
      out_string = f'{DIRECTORY}/{model}.{scenario}.{str(subset_time[0])}-{str(subset_time[1])}_processed.zarr'

      print("\nProcessing complete!")
      result.to_zarr(out_string, compute = True, zarr_format=2, consolidated=False, mode = 'w')
      print(f"{out_string} saved!")

      #Delete Raw Files
      file_pattern = f'/content/*.nc'
      files = glob.glob(file_pattern)
      for file in files:
        os.remove(file)
    except:
      print(f"Skipping {model}")
      continue

## LOCA

In [ ]:
from pathlib import Path

loca_directory = "https://gdo-dcp.llnl.gov/downscaled_cmip_projections/dcp/archive/cmip5/loca_hydro/LOCA_VIC_dpierce_2017-02-28"
#Get all data files for a model
for model in config.MODELS_LOCA:
  for scen in config.SCENARIOS['LOCA']:

    directory = f"{loca_directory}/{model}/vic_output.{scen}.netcdf/"
    #Accept (-A) only precipitation (pr) directories
    #Save to a temp file
    !wget -P /content/ -r -nc -np -A "precip.*.v0.nc" -nH --cut-dirs=3 -R "index.html*" {directory}

    directory_local = Path(f'/content/cmip5/loca_hydro/LOCA_VIC_dpierce_2017-02-28/{model}/vic_output.{scen}.netcdf/')

    i=0
    for file_path in directory_local.rglob("*"):  # recursively search
      if file_path.suffix in [".nc"]:
        #Clip
        ds_check = xr.open_dataset(file_path)
        ds_check = ds_check.rename({"Lat": "lat", "Lon": "lon"})
        ds_check = clip_domain(ds_check, ver = 'CMIP5')
        #Concat
        crs = ds_check.spatial_ref   # save CRS info
        if i==0:
          result = ds_check
        else:
          result = xr.concat([ds_check, result], dim = 'Time')
        ds_check.close()
        os.remove(file_path)
        i=i+1
    #Save
    result.to_netcdf(f'/content/drive/MyDrive/Research/MARISA_IDF/data/LOCA/LOCA_{model}_{scen}.nc')



# Calculate Adjustment Factors

### Small Example

In [ ]:
suffix = 'test'
ver = 'LOCA2'
if ver == 'LOCA2':
  DIRECTORY = "/content/drive/MyDrive/Research/MARISA_IDF/data/LOCA2/MARISA/"
  SAVE_DIR = "/content/drive/MyDrive/Research/MARISA_IDF/data/FINAL/LOCA2"
  scenarios = ['ssp245', 'ssp370', 'ssp585']
  models = config.MODELS_LOCA2

model = models[0]
scenario = scenarios[0]
subset_time = [2050, 2100]



r, r_mod, gev_mod = process_rp.calc_adj_factors(model, scenario, subset_time, DIRECTORY, n_b = 2, s_var = 'pds_peak_values', regionalization = True)

## Process Models Individually
Parallelize across models, scenarios, etc

In [ ]:
#Name your analysis
suffix = 'test_nb100_1950_2000'

#Select a Version
#ver = 'LOCA2'
ver = 'LOCA'

###################
if ver == 'LOCA2':
  DIRECTORY = "/content/drive/MyDrive/Research/MARISA_IDF/data/LOCA2/MARISA/"
  SAVE_DIR = "/content/drive/MyDrive/Research/MARISA_IDF/data/FINAL/LOCA2"
  scenarios = ['ssp245', 'ssp370', 'ssp585']
  models = config.MODELS_LOCA2
elif ver == 'LOCA':
  DIRECTORY = "/content/drive/MyDrive/Research/MARISA_IDF/data/LOCA/"
  SAVE_DIR = "/content/drive/MyDrive/Research/MARISA_IDF/data/FINAL/LOCA"
  scenarios = ['rcp45', 'rcp85']
  models = config.MODELS_LOCA

for scenario in scenarios:
    for model in models:
      # Log Precipitation Returns
      ds_returnPrecips_bootstrap = xr.Dataset(
          attrs=dict(description="Return Precipitation Threshold w PDS Bootstrap n=100"),
      )
      # Log Adjustment Factors
      ds_adjFactors_bootstrap = xr.Dataset(
          attrs=dict(description="Adjustment Factor w PDS Bootstrap n=100"),
      )

      # Log GEV Parameters
      ds_gevParas_bootstrap = xr.Dataset(
          attrs=dict(description="GEV Parameters w PDS Bootstrap n=100"),
      )

      print(f'Processing: {model}')
      for subset_time in [[2050, 2100], [2020, 2070]]:
        try:
          # r: Change Factors
          # r_mod: Depths
          # gev_mod: GEV Parameters (n_b = 0: baseline parameters)
          r, r_mod, gev_mod = process_rp.calc_adj_factors(model, scenario, subset_time, DIRECTORY, regionalization = True)
          flag = True
        except:
          print("model data not found")
          flag = False
          continue

        ds_adjFactors_bootstrap = xr.merge([ds_adjFactors_bootstrap, r], compat='no_conflicts', join = 'outer')
        ds_returnPrecips_bootstrap = xr.merge([ds_returnPrecips_bootstrap, r_mod], compat='no_conflicts', join = 'outer')
        ds_gevParas_bootstrap = xr.merge([ds_gevParas_bootstrap, gev_mod], compat='no_conflicts', join = 'outer')
      ### SAVE DATA ###
      if flag:
        ds_adjFactors_bootstrap.reset_index("county").to_zarr(f'{SAVE_DIR}/adj_factors_{ver}_{model}_{scenario}_{suffix}.zarr', zarr_format=2, consolidated=False, mode = 'w')
        ds_returnPrecips_bootstrap.reset_index("county").to_zarr(f'{SAVE_DIR}/return_precip_{ver}_{model}_{scenario}_{suffix}.zarr', zarr_format=2, consolidated=False, mode = 'w')
        ds_gevParas_bootstrap.reset_index("county").to_zarr(f'{SAVE_DIR}/gevParas_{ver}_{model}_{scenario}_{suffix}.zarr', zarr_format=2, consolidated=False, mode = 'w')

Processing: ACCESS1-0
Processing: ACCESS1-3
Processing: CCSM4
Processing: CESM1-BGC
Processing: CESM1-CAM5
Processing: CMCC-CM
Processing: CMCC-CMS
Processing: CNRM-CM5
Processing: CSIRO-Mk3-6-0
Processing: CanESM2
Processing: EC-EARTH
Processing: FGOALS-g2
Processing: GFDL-CM3
Processing: GFDL-ESM2G
Processing: GFDL-ESM2M
Processing: GISS-E2-H
model data not found
model data not found
Processing: GISS-E2-R


## Combine Individual Models into Single Xarray

In [ ]:
import os
ver = 'LOCA2'
#ver = 'LOCA'
if ver == 'LOCA2':
  DIRECTORY = "/content/drive/MyDrive/Research/MARISA_IDF/data/FINAL/LOCA2"
  SAVE_DIR = "/content/drive/MyDrive/Research/MARISA_IDF/data/FINAL/LOCA2"
  scenarios = ['ssp245', 'ssp370', 'ssp585']
  models = config.MODELS_LOCA2
elif ver == 'LOCA':
  DIRECTORY = "/content/drive/MyDrive/Research/MARISA_IDF/data/FINAL/LOCA/"
  SAVE_DIR = "/content/drive/MyDrive/Research/MARISA_IDF/data/FINAL/LOCA/"
  scenarios = ['rcp45', 'rcp85']
  models = config.MODELS_LOCA
FINAL_DIR = "/content/drive/MyDrive/Research/MARISA_IDF/data/FINAL/FINAL_COMBINED"

suffix = 'lmom_smoothedcentroid_nb100'

zarr_vars = ['adj_factor', 'return_precip', 'GEV_paras']
save_vars = ['adj_factors', 'return_precip', 'gevParas']
for save_var, zarr_var in list(zip(save_vars, zarr_vars)):
  temp = {}
  for scenario in scenarios:
    files = glob.glob(f'{SAVE_DIR}/{save_var}_{ver}_*_{scenario}_{suffix}.zarr')
    valid_files = []
    for file in files:
        zarr_path = os.path.join(file, zarr_var)
        if os.path.exists(zarr_path):  # Check if return_periods exists in zarr structure
            valid_files.append(file)
        else:
            print(f"Skipping {file} - no data")

    #Combine each data array by model

    temp[scenario] = xr.open_mfdataset(valid_files, combine='nested',
                                        concat_dim='model',
                                        consolidated=False,
                                        errors = 'ignore')

  if len(scenarios)==1:
    combined = temp[scenarios[0]]
  if len(scenarios)==2:
    combined = xr.concat([temp[scenarios[0]], temp[scenarios[1]]], dim='scenario')
  elif len(scenarios)>2:
    combined = xr.concat([temp[scenarios[0]], temp[scenarios[1]]], dim='scenario')
    combined = xr.concat([combined, temp[scenarios[2]]], dim='scenario')

  combined.to_zarr(f'{FINAL_DIR}/{save_var}_combined_{ver}_{suffix}.zarr', zarr_format=2, consolidated = False, mode='w')

/tmp/ipykernel_170/2134625554.py:44: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'model' ('model',) The recommendation is to set join explicitly for this case.
  combined = xr.concat([temp[scenarios[0]], temp[scenarios[1]]], dim='scenario')
/tmp/ipykernel_170/2134625554.py:45: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'model' ('model',) The recommendation is to set join explicitly for this case.
  combined = xr.concat([combined, temp[scenarios[2]]], dim='scenario')
/tmp/ipykernel_170/2134625554.py:44: Futur